## Vector RAG Testing using Chroma DB + Ollama

This is a code for testing Traditional RAG application on the legal document using Ollama + ChromaDB. Intention is to check its performance as compared to VectorLess RAG

Code by Saish Shetty. Dated 10/06/2026

Step 1: Install Dependencies

In [1]:
# pip install -q chromadb>=0.5.0 pymupdf>=1.24.0 langchain>=0.2.0 langchain-community>=0.2.0 ollama>=0.2.0 sentence-transformers>=3.0.0 langchain-text-splitters

Step 2: Document Ingestion and Embedding Generation

In [1]:
import os
import re
import fitz  # PyMuPDF
import chromadb
from chromadb.config import Settings
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

# ── Configuration ────────────────────────────────────────────────────────────
# PDF_PATH        = "document\constitutionofindiaacts.pdf"       # path to your PDF
PDF_PATH        = "document\Constitution_(131st_Amendment)_Bill,2026.pdf"       # path to your PDF
CHROMA_DIR      = "chroma_db"                     # persistent storage directory
COLLECTION_NAME = "delimitation_bill_india"
EMBED_MODEL     = "all-MiniLM-L6-v2"               # sentence-transformers model
CHUNK_SIZE      = 400                               # characters per chunk
CHUNK_OVERLAP   = 50                               # overlap between chunks
# ─────────────────────────────────────────────────────────────────────────────


def extract_text_from_pdf(pdf_path: str) -> list[dict]:
    """Extract text page-by-page from a PDF using PyMuPDF."""
    if not os.path.exists(pdf_path):
        raise FileNotFoundError(f"PDF not found: {pdf_path}")

    try:
        doc = fitz.open(pdf_path)
        pages = []
        for page_num in range(len(doc)):
            text = doc[page_num].get_text("text")
            # Basic cleanup
            text = re.sub(r'\n{3,}', '\n\n', text).strip()
            if text:
                pages.append({"page": page_num + 1, "text": text})
    except Exception as e:
        from pypdf import PdfReader
        reader = PdfReader(pdf_path)
        pages = []
        for page_num, page in enumerate(reader.pages, start=1):
            text = page.extract_text() or ""
            # Basic cleanup
            text = re.sub(r'\n{3,}', '\n\n', text).strip()
            if text:
                pages.append({"page": page_num + 1, "text": text})

    print(f"[✓] Extracted text from {len(pages)} pages.")
    return pages


def chunk_pages(pages: list[dict]) -> list[dict]:
    """Split page texts into smaller overlapping chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
        separators=["\n\n", "\n", ".", " "],
    )

    chunks = []
    for page in pages:
        splits = splitter.split_text(page["text"])
        for i, split in enumerate(splits):
            chunks.append({
                "text":     split,
                "page":     page["page"],
                "chunk_id": i,
            })

    print(f"[✓] Created {len(chunks)} chunks from {len(pages)} pages.")
    return chunks


def embed_texts(texts: list[str], model: SentenceTransformer) -> list[list[float]]:
    """Generate embeddings for a list of texts using sentence-transformers."""
    embeddings = model.encode(texts, show_progress_bar=True, batch_size=64)
    return embeddings.tolist()


def ingest():
    print("=" * 55)
    print("  Legal RAG — Ingestion Pipeline")
    print("=" * 55)

    # 1. Extract text from PDF
    pages = extract_text_from_pdf(PDF_PATH)

    # 2. Chunk the text
    chunks = chunk_pages(pages)

    # 3. Load sentence-transformers embedding model
    print(f"[→] Loading embedding model '{EMBED_MODEL}'...")
    embed_model = SentenceTransformer(EMBED_MODEL)

    # 4. Set up ChromaDB (persistent)
    client = chromadb.PersistentClient(
        path=CHROMA_DIR,
        settings=Settings(anonymized_telemetry=False),
    )

    # Drop existing collection to allow re-ingestion
    try:
        client.delete_collection(COLLECTION_NAME)
        print(f"[!] Existing collection '{COLLECTION_NAME}' deleted.")
    except Exception:
        pass

    collection = client.create_collection(
        name=COLLECTION_NAME,
        metadata={"hnsw:space": "cosine"},   # cosine similarity
    )

    # 5. Embed and store in batches
    BATCH_SIZE = 50
    total = len(chunks)
    print(f"\n[→] Embedding and storing {total} chunks in batches of {BATCH_SIZE}...")

    for start in range(0, total, BATCH_SIZE):
        batch     = chunks[start : start + BATCH_SIZE]
        texts     = [c["text"] for c in batch]
        ids       = [f"pg{c['page']}_ch{c['chunk_id']}_{start + i}" for i, c in enumerate(batch)]
        metadatas = [{"page": c["page"], "chunk_id": c["chunk_id"]} for c in batch]

        embeddings = embed_texts(texts, embed_model)

        collection.add(
            ids=ids,
            documents=texts,
            embeddings=embeddings,
            metadatas=metadatas,
        )
        print(f"  Stored chunks {start + 1}-{min(start + BATCH_SIZE, total)} / {total}")

    print(f"\n[✓] Ingestion complete! {total} chunks stored in '{CHROMA_DIR}'.")
# if __name__ == "__main__":

c:\Users\shett\miniconda3\envs\env3.11\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ingest()

  Legal RAG — Ingestion Pipeline
[✓] Extracted text from 10 pages.
[✓] Created 70 chunks from 10 pages.
[→] Loading embedding model 'all-MiniLM-L6-v2'...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1431.85it/s]



[→] Embedding and storing 70 chunks in batches of 50...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.69s/it]


  Stored chunks 1-50 / 70


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.01s/it]

  Stored chunks 51-70 / 70

[✓] Ingestion complete! 70 chunks stored in 'chroma_db'.


Step 3: Query on created chunks

In [ ]:
import chromadb
from chromadb.config import Settings
from ollama import Client as OllamaClient
from sentence_transformers import SentenceTransformer
import requests

# # ── Configuration ────────────────────────────────────────────────────────────
# CHROMA_DIR      = "chroma_db"
# COLLECTION_NAME = "constitution_india"
# EMBED_MODEL     = "all-MiniLM-L6-v2"   # sentence-transformers model
LLM_MODEL       = "qwen3.5:latest"
OLLAMA_BASE_URL = "http://localhost:11434"
TOP_K           = 5          # number of context chunks to retrieve
MAX_TOKENS      = 1024       # max tokens for the LLM response
# # ─────────────────────────────────────────────────────────────────────────────


SYSTEM_PROMPT = """You are an expert legal assistant specializing in the 
Constitution of India. Answer questions accurately using only the provided 
context excerpts. Cite relevant Articles or Parts when applicable. 
If the answer is not found in the context, say so clearly — do not fabricate."""


def build_rag_prompt(question: str, context_chunks: list[dict]) -> str:
    """Assemble the final prompt with retrieved context."""
    context_text = ""
    for i, chunk in enumerate(context_chunks, 1):
        page = chunk["metadata"].get("page", "?")
        context_text += f"\n[Excerpt {i} — Page {page}]\n{chunk['document']}\n"

    return f"""Use the following excerpts from the Constitution of India to answer the question.

--- CONTEXT ---
{context_text.strip()}
--- END CONTEXT ---

Question: {question}

Answer:"""


def retrieve(question: str, collection, embed_model: SentenceTransformer) -> list[dict]:
    """Embed the question and retrieve the top-K relevant chunks."""
    query_emb = embed_model.encode([question])[0].tolist()

    results = collection.query(
        query_embeddings=[query_emb],
        n_results=TOP_K,
        include=["documents", "metadatas", "distances"],
    )

    chunks = []
    for doc, meta, dist in zip(
        results["documents"][0],
        results["metadatas"][0],
        results["distances"][0],
    ):
        chunks.append({"document": doc, "metadata": meta, "distance": dist})

    return chunks


# def generate(prompt: str, ollama: OllamaClient) -> str:
#     """Call the Ollama LLM with the assembled RAG prompt."""
#     try:
#         print(f"[→] Sending prompt to {LLM_MODEL} ({len(prompt)} chars)...")
#         response = ollama.chat(
#             model=LLM_MODEL,
#             messages=[
#                 {"role": "system", "content": SYSTEM_PROMPT},
#                 {"role": "user",   "content": prompt},
#             ],
#             options={"num_predict": MAX_TOKENS, "temperature": 0.2},
#             stream=False,
#         )
#         print(f"[✓] Response received: {type(response)}")
#         print(f"[DEBUG] Response keys: {response.keys() if isinstance(response, dict) else 'N/A'}")
        
#         if isinstance(response, dict) and "message" in response:
#             return response["message"]["content"]
#         else:
#             print(f"[✗] Unexpected response format: {response}")
#             return None
#     except Exception as e:
#         print(f"[✗] Ollama error: {type(e).__name__}: {e}")
#         return None


def generate_rest(prompt: str) -> str:
    """Call Ollama REST API with proper timeout and error handling."""
    try:
        print(f"[→] Calling {LLM_MODEL} via REST API (timeout=60s)...")
        # print(f"[→] Prompt: {prompt}")
        
        response = requests.post(
            f"{OLLAMA_BASE_URL}/api/chat",
            json={
                "model": LLM_MODEL,
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": prompt},
                ],
                "stream": False,
                "think": False,
                "options": {"num_predict": MAX_TOKENS, "temperature": 0.2},
            },
            timeout=300  # Explicit timeout
        )
        
        if response.status_code != 200:
            print(f"[✗] Ollama returned status {response.status_code}: {response.text}")
            return None
        
        result = response.json()
        answer = result.get("message", {}).get("content", "")
        print(f"[✓] Got response ({len(answer)} chars)")
        return answer
        
    except requests.exceptions.Timeout:
        print(f"[✗] TIMEOUT: Ollama took >60 seconds. Model might be slow.")
        return "Error: Request timeout"
    except requests.exceptions.ConnectionError:
        print(f"[✗] Cannot connect to Ollama at {OLLAMA_BASE_URL}")
        return "Error: Ollama not reachable"
    except Exception as e:
        print(f"[✗] Error: {type(e).__name__}: {e}")
        return None

# def ask(question: str, collection, embed_model: SentenceTransformer, ollama: OllamaClient) -> str:
#     """Full RAG pipeline: retrieve → prompt → generate."""
#     print(f"\n[→] Retrieving top-{TOP_K} context chunks...")
#     chunks = retrieve(question, collection, embed_model)

#     print(f"[→] Generating answer with {LLM_MODEL}...")
#     prompt = build_rag_prompt(question, chunks)
#     answer = generate(prompt, ollama)

#     # Show source pages for transparency
#     pages = sorted({c["metadata"].get("page", "?") for c in chunks})
#     print(f"[i] Sources: pages {pages}")
#     print(chunks)

#     return answer


def ask(question: str, collection, embed_model: SentenceTransformer) -> str:
    """Full RAG pipeline: retrieve → prompt → generate."""
    print(f"\n[→] Retrieving top-{TOP_K} context chunks...")
    chunks = retrieve(question, collection, embed_model)
    
    print(f"[→] Building prompt...")
    prompt = build_rag_prompt(question, chunks)
    
    print(f"[→] Generating answer with {LLM_MODEL}...")
    answer = generate_rest(prompt)  # Use REST instead
    
    pages = sorted({c["metadata"].get("page", "?") for c in chunks})
    print(f"[i] Sources: pages {pages}")
    
    return answer if answer else "No answer generated"


def interactive_loop(collection, embed_model: SentenceTransformer, ollama: OllamaClient):
    """Simple REPL for interactive Q&A."""
    print("\n" + "=" * 55)
    print("  Legal RAG — Constitution of India Q&A")
    print("  Type 'exit' or 'quit' to stop.")
    print("=" * 55)

    while True:
        question = input("\nYour question: ").strip()
        if not question:
            continue
        if question.lower() in {"exit", "quit"}:
            print("Goodbye!")
            break

        answer = ask(question, collection, embed_model, ollama)
        print(f"\nAnswer:\n{'-' * 40}\n{answer}\n{'-' * 40}")


def main():
    # Connect to persisted ChromaDB
    client = chromadb.PersistentClient(
        path=CHROMA_DIR,
        settings=Settings(anonymized_telemetry=False),
    )

    try:
        collection = client.get_collection(COLLECTION_NAME)
        print(f"[✓] Connected to collection '{COLLECTION_NAME}' "
            f"({collection.count()} chunks).")
    except Exception:
        print(f"[✗] Collection '{COLLECTION_NAME}' not found.")
        print("    Please run 'python ingest.py' first.")
        return

    # Load embedding model (same one used during ingestion)
    print(f"[→] Loading embedding model '{EMBED_MODEL}'...")
    embed_model = SentenceTransformer(EMBED_MODEL)

    ollama = OllamaClient(host=OLLAMA_BASE_URL)

    # ── Example single query (uncomment to test) ──────────────────────────
    # answer = ask("What are the Fundamental Rights in the Constitution?",
    #             collection, embed_model)
    # answer = ask("What are the Fundamental Duties of Indian Citizens?",
    #             collection, embed_model)
    answer = ask("What is the provision for women in the 131st amendment bill?",
                collection, embed_model)
    print(answer)
    # ──────────────────────────────────────────────────────────────────────

    # interactive_loop(collection, embed_model, ollama)

In [6]:
main()

[✓] Connected to collection 'delimitation_bill_india' (70 chunks).
[→] Loading embedding model 'all-MiniLM-L6-v2'...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4424.78it/s]



[→] Retrieving top-5 context chunks...
[→] Building prompt...
[→] Generating answer with qwen3.5:latest...
[→] Calling qwen3.5:latest via REST API (timeout=60s)...
[✓] Got response (1525 chars)
[i] Sources: pages [4, 5]
Based on the provided context excerpts, there is **no mention of a "131st Amendment Bill."**

The text explicitly discusses the **"Constitution (Eighty-fourth Amendment) Act, 2001"** and refers to a **"proposed Bill"** intended to amend specific articles. The provisions described in this proposed bill include:

*   **Amendments:** Inserting Articles **330A**, **332A**, and **334A** into the Constitution (Excerpt 3).
*   **Objective:** To provide for reservation of nearly one-third of seats for women in the House of the People, Legislative Assemblies, and the National Capital Territory of Delhi. This includes reservations for women belonging to Scheduled Castes and Scheduled Tribes (Excerpts 1, 2, and 3).
*   **Operationalization:** The reservation is intended to become

#### <SIDENOTE> Ollama Testing Code

In [13]:
# import requests

# # Test 1: Can we reach Ollama?
# print("[→] Testing Ollama connectivity...")
# try:
#     response = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=5)
#     models = response.json()
#     print(f"[✓] Ollama is reachable. Loaded models: {[m['name'] for m in models.get('models', [])]}")
# except Exception as e:
#     print(f"[✗] Cannot reach Ollama: {e}")

# # Test 2: Simple REST call with timeout
# print("\n[→] Testing simple REST call to qwen3.5...")
# test_payload = {
#     "model": LLM_MODEL,
#     "messages": [{"role": "user", "content": "Say 'Hello' in one word"}],
#     "stream": False,
# }
# try:
#     response = requests.post(
#         f"{OLLAMA_BASE_URL}/api/chat", 
#         json=test_payload, 
#         timeout=120  # Critical: 30-second timeout
#     )
#     result = response.json()
#     print(f"[✓] Simple test successful: {result.get('message', {}).get('content', 'No response')}")
# except requests.exceptions.Timeout:
#     print(f"[✗] TIMEOUT: Ollama took too long (>30s). Model may be slow or overloaded.")
# except Exception as e:
#     print(f"[✗] REST call failed: {e}")